In [6]:
# ============================================================
# 02 Baseline Random Forest Results Analysis
# ============================================================

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    roc_curve,
    precision_recall_curve,
    auc,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize

from lif_thesis.models.baseline_rf import build_spectra_matrix

In [7]:
# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ROOT = Path.cwd().parents[1]

DATA_PATH = ROOT / "data" / "processed" / "bacterial_samples.parquet"
RESULTS_DIR = ROOT / "results" / "baseline_rf_multimodal"

METRICS_PATH = RESULTS_DIR / "metrics.json"
MODEL_PATH = RESULTS_DIR / "baseline_rf_multimodal.joblib"
ENCODER_PATH = RESULTS_DIR / "label_encoder.joblib"

print("Root:", ROOT)
print("Data path:", DATA_PATH)
print("Results path:", RESULTS_DIR)

Root: c:\Users\chris\OneDrive\Documents\Universitat de Barcelona\Automatic-Recognition-of-Microbial-Life
Data path: c:\Users\chris\OneDrive\Documents\Universitat de Barcelona\Automatic-Recognition-of-Microbial-Life\data\processed\bacterial_samples.parquet
Results path: c:\Users\chris\OneDrive\Documents\Universitat de Barcelona\Automatic-Recognition-of-Microbial-Life\results\baseline_rf_multimodal


In [8]:
# ------------------------------------------------------------
# 2. Load data, model, metrics, and split indices
# ------------------------------------------------------------

df = pd.read_parquet(DATA_PATH)

model = joblib.load(MODEL_PATH)
label_encoder = joblib.load(ENCODER_PATH)

with open(METRICS_PATH, "r") as f:
    metrics = json.load(f)

train_idx = np.load(RESULTS_DIR / "train_idx.npy")
val_idx = np.load(RESULTS_DIR / "val_idx.npy")
test_idx = np.load(RESULTS_DIR / "test_idx.npy")

print("Data shape:", df.shape)
print("Classes:", label_encoder.classes_)

Data shape: (239638, 40)
Classes: ['B. cereus' 'B. endophyticus' 'K. salsicia' 'M. luteus' 'S. huminis']


In [9]:
# ------------------------------------------------------------
# 3. Summarize split composition
# ------------------------------------------------------------

split_summary = []

for split_name, idx in {
    "train": train_idx,
    "validation": val_idx,
    "test": test_idx,
}.items():
    split_df = df.iloc[idx]

    split_summary.append({
        "split": split_name,
        "n_particles": len(split_df),
        "n_raw_files": split_df["raw_file"].nunique(),
        "n_species": split_df["species"].nunique(),
    })

split_summary_df = pd.DataFrame(split_summary)
display(split_summary_df)

for split_name, idx in {
    "train": train_idx,
    "validation": val_idx,
    "test": test_idx,
}.items():
    print(f"\n{split_name.upper()} species distribution")
    display(df.iloc[idx]["species"].value_counts(normalize=True))

,split,n_particles,n_raw_files,n_species
0,train,167747,35,5
1,validation,47926,10,5
2,test,23965,5,5



TRAIN species distribution


species
K. salsicia        0.200033
B. cereus          0.200027
S. huminis         0.200010
M. luteus          0.199974
B. endophyticus    0.199956
Name: proportion, dtype: float64


VALIDATION species distribution


species
B. cereus          0.200038
K. salsicia        0.200038
M. luteus          0.200017
S. huminis         0.199975
B. endophyticus    0.199933
Name: proportion, dtype: float64


TEST species distribution


species
B. cereus          0.200042
K. salsicia        0.200042
S. huminis         0.200000
B. endophyticus    0.199958
M. luteus          0.199958
Name: proportion, dtype: float64

In [10]:
# ------------------------------------------------------------
# 4. Compact metrics table
# ------------------------------------------------------------

rows = []

for split_name in ["train", "val", "test"]:
    m = metrics[split_name]

    rows.append({
        "split": split_name,
        "accuracy": m.get("accuracy"),
        "balanced_accuracy": m.get("balanced_accuracy"),
        "macro_f1": m.get("macro_f1"),
        "weighted_f1": m.get("weighted_f1"),
        "roc_auc": m.get("roc_auc"),
        "pr_auc": m.get("pr_auc"),
    })

metrics_df = pd.DataFrame(rows)
display(metrics_df)

,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,roc_auc,pr_auc
0,train,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,val,0.861703,0.861723,0.859845,0.859827,0.970424,0.898689
2,test,0.873983,0.874005,0.872739,0.872719,0.974024,0.912859
